REMOVING OF URL CONTAINING THE ID

In [1]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/processed/cleaned.csv")

print("Shape:", df.shape)
df.head()


C:\Users\DELL\AppData\Local\Temp\ipykernel_9092\337512669.py:4: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned.csv")


Shape: (2260701, 112)


,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag
0,68407277,3600.0,3600.0,3600.0,36.0,13.99,123.03,C,C4,leadman,...,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,N,Cash,N
1,68355089,24700.0,24700.0,24700.0,36.0,11.99,820.28,C,C1,Engineer,...,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,N,Cash,N
2,68341763,20000.0,20000.0,20000.0,60.0,10.78,432.66,B,B4,truck driver,...,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0,N,Cash,N
3,66310712,35000.0,35000.0,35000.0,60.0,14.85,829.90,C,C5,Information Systems Officer,...,0.0,0.0,0.0,381215.0,52226.0,62500.0,18000.0,N,Cash,N
4,68476807,10400.0,10400.0,10400.0,60.0,22.45,289.91,F,F1,Contract Specialist,...,60.0,0.0,0.0,439570.0,95768.0,20300.0,88097.0,N,Cash,N


In [2]:
"url" in df.columns


True

In [3]:
# Drop URL column
df = df.drop(columns=["url"])

print("URL removed")
"url" in df.columns


URL removed


False

In [4]:
[col for col in df.columns if "url" in col.lower()]


[]

INCOME GENERALISATION- BUCKETING

In [6]:
df['annual_inc'].describe()


count    2.260701e+06
mean     7.799222e+04
std      1.126953e+05
min      0.000000e+00
25%      4.600000e+04
50%      6.500000e+04
75%      9.300000e+04
max      1.100000e+08
Name: annual_inc, dtype: float64

In [7]:
import numpy as np

bins = [0, 40000, 60000, 80000, 100000, np.inf]
labels = ["<40k", "40k-60k", "60k-80k", "80k-100k", "100k+"]

df['income_band'] = pd.cut(
    df['annual_inc'],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [8]:
df[['annual_inc','income_band']].head(10)


,annual_inc,income_band
0,55000.0,40k-60k
1,65000.0,60k-80k
2,63000.0,60k-80k
3,110000.0,100k+
4,104433.0,100k+
5,34000.0,<40k
6,180000.0,100k+
7,85000.0,80k-100k
8,85000.0,80k-100k
9,42000.0,40k-60k


In [9]:
df['income_band'].value_counts(normalize=True)*100


income_band
40k-60k     26.821150
60k-80k     21.362312
100k+       19.779396
<40k        18.186571
80k-100k    13.850571
Name: proportion, dtype: float64

In [10]:
df = df.drop(columns=['annual_inc'])

print("annual_inc removed")
'annual_inc' in df.columns


annual_inc removed


False

FICO SCORE BUCKETING

In [11]:
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
df['fico_avg'].describe()


count    2.260701e+06
mean     7.005882e+02
std      3.301059e+01
min      6.120000e+02
25%      6.770000e+02
50%      6.920000e+02
75%      7.170000e+02
max      8.475000e+02
Name: fico_avg, dtype: float64

In [12]:
bins = [0, 580, 670, 740, 800, float("inf")]
labels = ["Poor", "Fair", "Good", "Very Good", "Excellent"]

df["fico_band"] = pd.cut(
    df["fico_avg"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [13]:
df[["fico_avg", "fico_band"]].head(10)


,fico_avg,fico_band
0,677.0,Good
1,717.0,Good
2,697.0,Good
3,787.0,Very Good
4,697.0,Good
5,692.0,Good
6,682.0,Good
7,707.0,Good
8,687.0,Good
9,702.0,Good


In [14]:
df["fico_band"].value_counts(normalize=True) * 100


fico_band
Good         71.252280
Fair         16.270528
Very Good    11.103812
Excellent     1.373379
Poor          0.000000
Name: proportion, dtype: float64

In [15]:
df = df.drop(columns=[
    "fico_range_low",
    "fico_range_high",
    "fico_avg"
])

[col for col in df.columns if "fico" in col.lower()]



['last_fico_range_high', 'last_fico_range_low', 'fico_band']

In [ ]:
df = df.drop(columns=[
    "last_fico_range_high",
    "last_fico_range_low"
])

print("Last FICO columns removed")

[col for col in df.columns if "fico" in col.lower()]

#last_fico is also removed because if it is given, then the model learns the outcome
#and it is data leakage 

REDUCE TEMPORAL PRECISION (DATES-->YEARS/DURATIONS)

In [18]:
# Convert issue_d to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%y', errors='coerce')

# Extract year
df['issue_year'] = df['issue_d'].dt.year


In [19]:
# Convert earliest credit line
df['earliest_cr_line'] = pd.to_datetime(
    df['earliest_cr_line'],
    format='%b-%y',
    errors='coerce'
)

# Credit history length in years
df['credit_history_years'] = (
    df['issue_d'].dt.year - df['earliest_cr_line'].dt.year
)


In [20]:
df = df.drop(columns=[
    'issue_d',
    'earliest_cr_line',
    'last_pymnt_d',
    'next_pymnt_d',
    'last_credit_pull_d'
], errors='ignore')


LOAN AMOUNT BUCKETING

In [21]:
df['loan_amnt'].describe()


count    2.260701e+06
mean     1.504690e+04
std      9.190182e+03
min      5.000000e+02
25%      8.000000e+03
50%      1.290000e+04
75%      2.000000e+04
max      4.000000e+04
Name: loan_amnt, dtype: float64

In [22]:
import numpy as np

bins = [0, 5000, 10000, 20000, 30000, np.inf]
labels = ["<5k", "5k-10k", "10k-20k", "20k-30k", "30k+"]

df['loan_band'] = pd.cut(
    df['loan_amnt'],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [23]:
df[['loan_amnt', 'loan_band']].head(10)
df['loan_band'].value_counts(normalize=True)*100
df = df.drop(columns=['loan_amnt'])


INSTALLMENT (EMI) BUCKETING

In [24]:
df['installment'].describe()


count    2.260701e+06
mean     4.458058e+02
std      2.671717e+02
min      4.930000e+00
25%      2.516500e+02
50%      3.779900e+02
75%      5.933200e+02
max      1.719830e+03
Name: installment, dtype: float64

In [25]:
import numpy as np

bins = [0, 200, 350, 600, 900, np.inf]
labels = ["<200", "200-350", "350-600", "600-900", "900+"]

df['emi_band'] = pd.cut(
    df['installment'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df['emi_band'].value_counts(normalize=True)*100


emi_band
350-600    30.404153
200-350    28.306043
600-900    17.244518
<200       16.841900
900+        7.203385
Name: proportion, dtype: float64

In [26]:
df = df.drop(columns=['installment'])


GENERALZING THE AVERAGE_CURRENT_BALANCE

In [27]:
df['avg_cur_bal'].describe()


count    2.260701e+06
mean     1.335438e+04
std      1.625147e+04
min      0.000000e+00
25%      3.163000e+03
50%      7.335000e+03
75%      1.829100e+04
max      9.580840e+05
Name: avg_cur_bal, dtype: float64

In [28]:
import numpy as np

# Top-code extreme values
df['avg_cur_bal_capped'] = df['avg_cur_bal'].clip(upper=100000)

# Create balance bands
bins = [0, 2000, 5000, 10000, 20000, 100000, np.inf]
labels = ["<2k", "2k-5k", "5k-10k", "10k-20k", "20k-100k", "100k+"]

df['bal_band'] = pd.cut(
    df['avg_cur_bal_capped'],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [29]:
df['bal_band'].value_counts(normalize=True)*100
df = df.drop(columns=['avg_cur_bal', 'avg_cur_bal_capped'])


SAVE THE SECURED DATA

In [30]:
import os

# Secure folder path
SECURE_DIR = "../data/secure"
SECURE_FILE = "secure_ml_dataset.csv"

# Create folder if not exists
os.makedirs(SECURE_DIR, exist_ok=True)

# Full path
secure_path = os.path.join(SECURE_DIR, SECURE_FILE)

# Save
df.to_csv(secure_path, index=False)

print("Secure dataset saved at:")
print(secure_path)


Secure dataset saved at:
../data/secure\secure_ml_dataset.csv
